# 第二章：练习答案

[回到主教程](ch02.ipynb)。本子可独立运行，只用小数组，不加载语言模型。
这里导入冻结版本的组件作为答案；主教程的真实验证使用主教程当前定义的类。

In [1]:
from pathlib import Path
import sys
import mlx.core as mx
import numpy as np

here = Path.cwd().resolve()
chapter_dir = next(p for p in [here, here / 'ch02', here / 'notebooks/ch02']
                   if (p / 'reference/layers.py').is_file())
sys.path.insert(0, str(chapter_dir / 'reference'))
from layers import Embedding, Linear, RMSNorm, MLP, FeedForwardBlock

## 1. 重复 ID 会查到同一行

三个 ID、每行三个数，输出形状为 `[3,3]`。第一个和第三个 ID 都是0，所以对应行相同。

In [2]:
weight = mx.array([[1., 0., 1.], [0., 1., 1.], [1., 1., 0.]])
actual = Embedding(weight)(mx.array([0, 2, 0]))
assert actual.shape == (3, 3)
np.testing.assert_array_equal(np.array(actual[0]), np.array(actual[2]))
print(np.array(actual))

[[1. 0. 1.]
 [1. 1. 0.]
 [1. 0. 1.]]


## 2. w=0，不代表缩放系数为0

对这个模型，缩放系数是 `1+w`。漏掉1，输出会变成零向量。

In [3]:
x = mx.array([[3., 4.]])
stored_weight = mx.zeros(2)
correct = RMSNorm(stored_weight, eps=0.)(x)
wrong = RMSNorm(stored_weight, eps=0., zero_centered=False)(x)
print('正确:', np.array(correct))
print('漏掉1:', np.array(wrong))
np.testing.assert_array_equal(np.array(wrong), [[0., 0.]])
assert not np.allclose(np.array(correct), np.array(wrong))

正确: [[0.84852815 1.1313709 ]]
漏掉1: [[0. 0.]]


## 3. 残差必须加回原始 x

让 down 投影为零，MLP 输出就为零。正确结果是 x；错加 `norm(x)` 得到的会是归一化结果。
这项检查能发现“形状完全正确、数值逻辑却错了”的实现。

In [4]:
norm = RMSNorm(mx.zeros(2))
identity = Linear(mx.eye(2))
mlp = MLP(identity, identity, Linear(mx.zeros((2, 2))))
correct = FeedForwardBlock(norm, mlp)(x)
wrong = norm(x) + mlp(norm(x))
np.testing.assert_array_equal(np.array(correct), np.array(x))
assert not np.allclose(np.array(wrong), np.array(x))
print('正确残差:', np.array(correct))
print('错误残差:', np.array(wrong))

正确残差: [[3. 4.]]
错误残差: [[0.8485281 1.1313708]]
